Использование двух двух классовых моделей для вывода

In [ ]:
import os
import numpy as np
from PIL import Image
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import load_model

In [ ]:
# 1. Загрузка моделей
model1 = load_model('/content/best_model.h5')
model2 = load_model('/content/best_model_VGG.h5')

In [ ]:
# 2.  Определение классов
classes = ['all', 'noall']  # Оба классификатора используют одни и те же классы
TEST_DIR = '/content/test' #Директория с данными

# 3. Функции для загрузки и предобработки изображений
def load_and_preprocess_image_grayscale(image_path, target_size=(400, 400)):  # для model1 (grayscale)
    """Загружает изображение, изменяет размер и нормализует для grayscale модели."""
    try:
        img = Image.open(image_path).convert('L')  # Преобразуем в оттенки серого
        img = img.resize(target_size)
        img_array = np.array(img) / 255.0  # Нормализация
        img_array = np.expand_dims(img_array, axis=0)  # Добавляем размерность для батча
        img_array = np.expand_dims(img_array, axis=-1) # Добавляем канал
        return img_array
    except Exception as e:
        print(f"Ошибка при загрузке изображения {image_path}: {e}")
        return None


def load_and_preprocess_image_rgb(image_path, target_size=(224, 224)):  # для model2 (RGB)
    """Загружает изображение, изменяет размер и нормализует для RGB модели."""
    try:
        img = Image.open(image_path).convert('RGB')
        img = img.resize(target_size)
        img_array = np.array(img) / 255.0  # Нормализация
        img_array = np.expand_dims(img_array, axis=0)  # Добавляем размерность для батча
        return img_array
    except Exception as e:
        print(f"Ошибка при загрузке изображения {image_path}: {e}")
        return None

In [ ]:
# 4. Функция для классификации изображения
def classify_image(image_path, model1, model2):
    """Классифицирует изображение обеими моделями."""
    img_array_grayscale = load_and_preprocess_image_grayscale(image_path)
    img_array_rgb = load_and_preprocess_image_rgb(image_path)

    if img_array_grayscale is None or img_array_rgb is None:
        return None, None

    # Classification model 1 (grayscale)
    prediction1 = model1.predict(img_array_grayscale)[0]  # TensorFlow/Keras
    # with torch.no_grad(): # PyTorch inference context
    #     img_tensor = transforms.Grayscale()(transforms.ToTensor()(img)).unsqueeze(0)
    #     prediction1 = torch.sigmoid(model1(img_tensor)).numpy()[0]
    class_index1 = np.argmax(prediction1)
    class_name1 = classes[class_index1]
    confidence1 = prediction1[class_index1]

    # Classification model 2 (RGB)
    prediction2 = model2.predict(img_array_rgb)[0]  # TensorFlow/Keras
    # with torch.no_grad():
    #     img_tensor = transforms.ToTensor()(img).unsqueeze(0)
    #     prediction2 = torch.sigmoid(model2(img_tensor)).numpy()[0]
    class_index2 = np.argmax(prediction2)
    class_name2 = classes[class_index2]
    confidence2 = prediction2[class_index2]
    return (class_name1, confidence1, prediction1), (class_name2, confidence2, prediction2)



# 5.  Обработка тестовой выборки
def process_test_data(test_dir):
    """Классифицирует все изображения в тестовой директории и сравнивает результаты."""
    true_labels = []
    predicted_labels = []
    filenames = []

    # Итерируемся по папкам классов all и noall
    for class_name in classes:
        class_dir = os.path.join(test_dir, class_name) #Путь к папке с определенным классом

        # Проверяем, что такая папка существует
        if not os.path.exists(class_dir):
            print(f"Внимание: папка '{class_dir}' не найдена.")
            continue #Если нет такой папки, переходим к следующей итерации

        # Получаем список файлов для каждого класса
        image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        # Проходим по файлам в папке класса
        for filename in image_files:
            image_path = os.path.join(class_dir, filename)
            filenames.append(filename)
            result1, result2 = classify_image(image_path, model1, model2) #Классифицируем изображение

            if result1 is None or result2 is None:
                print(f"Пропущено {filename} из-за ошибки загрузки.")
                continue # Пропускаем проблемное изображение

            class_name1, confidence1, _ = result1
            class_name2, confidence2, _ = result2

            # Сравниваем классификации
            if class_name1 == class_name2:
                print(f"Уверенная классификация для {filename}: {class_name1}")
                final_class = class_name1
            else:
                print(f"Несовпадение для {filename}. Model 1: {class_name1} ({confidence1:.4f}), Model 2: {class_name2} ({confidence2:.4f}).")
                # Выбираем класс с наибольшей уверенностью
                if confidence1 >= confidence2:
                    final_class = class_name1
                    print(f"Выбран класс из model 1: {final_class} с уверенностью {confidence1:.4f}") # Вывод уверенности
                else:
                    final_class = class_name2
                    print(f"Выбран класс из model 2: {final_class} с уверенностью {confidence2:.4f}") # Вывод уверенности

            # Ground truth label - используем имя папки, в которой находится изображение
            true_label = class_name  # Имя папки и есть ground truth label

            true_labels.append(true_label)
            predicted_labels.append(final_class)

    return true_labels, predicted_labels, filenames


# 6. Оценка результатов
def evaluate_results(true_labels, predicted_labels, agreement_labels):
    """Вычисляет метрики precision, recall, accuracy для реальных совпадений и выводит матрицу соответствия."""
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='weighted', zero_division=0)  # average = weighted нужно, если классы несбалансированы
    cm = confusion_matrix(true_labels, predicted_labels)

    # Вычисляем accuracy только для тех случаев, когда модели согласны (реальные совпадения)
    agreement_accuracy = accuracy_score(true_labels, predicted_labels, sample_weight=agreement_labels) #Передаем sample_weight
    print("\nМетрики:")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Agreement Accuracy: {agreement_accuracy:.4f}")  # Выводим accuracy для реальных совпадений
    print(f"Confusion Matrix:\n{cm}")



In [ ]:

# 7.  Основной код
test_directory = TEST_DIR  # Укажите путь к тестовой выборке (теперь он задан в TEST_DIR)
true_labels, predicted_labels, filenames = process_test_data(test_directory)

if true_labels and predicted_labels:
    evaluate_results(true_labels, predicted_labels)
else:
    print("Не удалось провести оценку: нет данных.")

print("Done!")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 715ms/step
Уверенная классификация для def9_9.jpg: all
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 680ms/step
Уверенная классификация для def74_19.jpg: all
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 798ms/step
Уверенная классификация для def14_1.jpg: all
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Уверенная классификация для def32_2.jpg: all
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Уверенная классификация для def46_4.jpg: all
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 699ms/step
Уверенная классификация для def53_16.jpg: all
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 683ms/step
Уверенная классификация для def64_1.jpg: all
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 708ms/step
Уверенная классификация для def41_5.jpg: all
1/1 ━━━━━━━━━━━━━━━━━━

TypeError: evaluate_results() missing 1 required positional argument: 'agreement_labels'

In [ ]:
# Размеры изображений для моделей
img_width1, img_height1 = 400, 400
img_width2, img_height2 = 224, 224

# Функция для предобработки изображения для первой модели
def preprocess_image_model1(img_path):
    img = image.load_img(img_path, target_size=(img_width1, img_height1), color_mode='grayscale')
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.  # Масштабирование
    return img_array

# Функция для предобработки изображения для второй модели
def preprocess_image_model2(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (img_width2, img_height2))

    # Преобразование в RGB, если изображение оттенков серого
    if len(img.shape) == 2 or img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.  # Масштабирование
    return img_array

# Функция для классификации и сравнения результатов
def classify_image(img_path):
    # Предобработка для первой модели
    img_array1 = preprocess_image_model1(img_path)
    # Получение предсказания от первой модели
    prediction1 = model1.predict(img_array1)
    class_index1 = np.argmax(prediction1)  # Получаем индекс класса с наибольшей вероятностью
    confidence1 = prediction1[0][class_index1] # Берем уверенность для предсказанного класса

    # Предобработка для второй модели
    img_array2 = preprocess_image_model2(img_path)
    # Получение предсказания от второй модели
    prediction2 = model2.predict(img_array2)
    class_index2 = np.argmax(prediction2) # Получаем индекс класса с наибольшей вероятностью
    confidence2 = prediction2[0][class_index2]  # Берем уверенность для предсказанного класса


    # Определяем классы (соответствовали порядку в train_generator)
    class_names = ['all', 'noall']  # Замените на ваши фактические имена классов

    # Сравнение результатов
    if class_index1 == class_index2:
        print(f"Уверенный результат: Объект классифицирован как '{class_names[class_index1]}' с уверенностью {max(confidence1, confidence2):.2f}")
    else:
        if confidence1 > confidence2:
            print(f"Результат с расхождением: Объект классифицирован как '{class_names[class_index1]}' с уверенностью {confidence1:.2f} (модель 1)")
        else:
            print(f"Результат с расхождением: Объект классифицирован как '{class_names[class_index2]}' с уверенностью {confidence2:.2f} (модель 2)")





In [ ]:
# одиночное изображение
image_path = '/content/sample_data/test5classes/noall_nodefects_image (9).jpg'  # Замените на путь к вашему изображению
classify_image(image_path)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Уверенный результат: Объект классифицирован как 'noall' с уверенностью 0.81


In [ ]:
# папка с изображениями
def classify_images_in_folder(folder_path):
    for filename in os.listdir(folder_path):
        if filename.endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff')):
            image_path = os.path.join(folder_path, filename)
            print(f"Классификация изображения: {filename}")
            classify_image(image_path)
            print("-" * 30)